In [ ]:
#Necessary Dependencies 
import sys
sys.path.append("..")

import importlib
import tensorflow as tf
import DataModules.DataAugmentation
importlib.reload(DataModules.DataAugmentation)

from tensorflow import keras
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Maximum, Input
from DataModules.DataAugmentation import get_datasets

In [ ]:
#Loading Data

train_dataset1, test_dataset1, train_dataset2, test_dataset2, train_dataset3, test_dataset3, train_dataset4, test_dataset4, data_augmentation = get_datasets()

combined_train = train_dataset1.concatenate(train_dataset2)
combined_train = combined_train.concatenate(train_dataset3)
combined_train = combined_train.concatenate(train_dataset4)

combined_test = test_dataset1.concatenate(test_dataset2)
combined_test = combined_test.concatenate(test_dataset3)
combined_test = combined_test.concatenate(test_dataset4)

In [ ]:
#LOADING THE NEEDED MODELS
GoogleNet = load_model('/content/drive/MyDrive/Colab Notebooks/Main Project/Saved Models/GoogleNet/googlenet.h5')
ResNet = load_model('/content/drive/MyDrive/Colab Notebooks/Main Project/Saved Models/ResNet/resnet.h5')
NASnet = load_model('/content/drive/MyDrive/Colab Notebooks/Main Project/Saved Models/NASnet/nasnet.h5')
DenseNet = load_model('/content/drive/MyDrive/Colab Notebooks/Main Project/Saved Models/Densenet/densenet.h5')

In [ ]:
#Combined Model

models = [GoogleNet, ResNet, NASNet, DenseNet]
model_input = Input(shape=(224, 224, 3))
model_output = [model(model_input) for model in models]
ensemble_output = Maximum()(model_output)

ensemble_2 = Model(inputs=model_input, outputs=ensemble_output)

In [ ]:
ensemble_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
    )

history = {}

history = ensemble_2.fit(
        combined_train,
        validation_data=combined_test,
        epochs=150
    )